# Step 4: Deploy MCP Server

Deploy the Claims MCP server (Athena tools) to AgentCore Runtime.

An **MCP server** exposes tools the agent calls through a gateway — here, claims queries (`query_claims`, `get_claim_details`, `get_claims_summary`, …) backed by Athena / S3 Tables. This is the **claims / Athena half** of the two-pattern story: the Claims Gateway (GW1) you build in `05a` fronts this server. Its queries aren't open — each call runs under a tenant IAM role, so **Lake Formation** governs which columns each persona sees, while the tools themselves bind the caller's identity into a `WHERE user_id = ?` predicate to scope rows.

## Prerequisites

- ✅ Run `03-deploy-s3tables.ipynb` first
- ✅ AgentCore CLI 0.30.0 installed (`npm install -g @aws/agentcore@0.30.0`, Node.js 20+) and `uv` on PATH

## What This Notebook Does

1. Creates the MCP server's execution role
2. Deploys MCP Server to AgentCore Runtime with the AgentCore CLI
3. Configures JWT authentication for the active IdP (Cognito or Okta)
4. Saves Runtime ARN to SSM

## Next Notebook

- **05a-deploy-claims-gateway.ipynb**

In [ ]:
# AWS Initialization - Load credentials and create session
from utils.notebook_init import init_aws
import os

# This will:
# 1. Load credentials from .env file (if it exists)
# 2. Create and validate AWS session (env vars take precedence over SSO)
# 3. Return session, region, and account_id for use in this notebook
session, region, account_id = init_aws()

# Initialize AWS clients
ssm_client = session.client("ssm", region_name=region)
os.environ["AWS_DEFAULT_REGION"] = region

print("✅ Ready to proceed with AWS operations")
print(f"   Account ID: {account_id}")
print(f"   Region: {region}")

## Step 1: Deploy MCP Server
First creates the server's execution role. It is created here rather than in notebook 02 because its policy is scoped to the S3 bucket that notebook 03 created.

Then deploys the server's AgentCore CLI project (`deployment/4a-mcp-lakehouse-server/agentcore/`) as its own CloudFormation stack. The CLI packages the source directory as a CodeZip; there is no container build.
⏱️ ~several min.

In [ ]:
import subprocess
import sys

# Create (or update) this runtime's execution role, AgentCoreRuntimeRole-lakehouse-mcp.
# Created here, not in notebook 02: its policy is scoped to the S3 bucket from notebook 03.
result = subprocess.run(
    [sys.executable, "setup_runtime_roles.py", "create", "--role", "lakehouse-mcp"],
    cwd="deployment/2-lakehouse-tenant-roles-setup",
)
if result.returncode != 0:
    print("❌ Error: setup_runtime_roles.py failed (see output above)")
else:
    print("\n✅ Execution role AgentCoreRuntimeRole-lakehouse-mcp ready")

In [ ]:
import subprocess
import sys

# Deploys this runtime's AgentCore CLI project (deployment/4a-mcp-lakehouse-server/agentcore/)
# as its own CloudFormation stack, then stores the runtime ARN/ID in SSM.
# No capture_output: stdout/stderr stream live to the notebook so the deploy
# progress appears incrementally instead of all at once when the script ends.
result = subprocess.run(
    [sys.executable, "agentcore_cli_deploy.py", "lakehouse-mcp"],
    cwd="deployment",
)
if result.returncode != 0:
    print("❌ Error: agentcore_cli_deploy.py failed (see output above)")
else:
    print("\n✅ MCP Server deployed!")
    print("\n📋 Runtime configuration saved to SSM Parameter Store")

## Step 2: Verify MCP Server Deployment

The `agentcore_cli_deploy.py` script automatically saves the Runtime ARN to SSM.
Run this cell to verify the deployment.

In [ ]:
# Verify MCP Server Runtime configuration in SSM
print("Verifying MCP Server Runtime in SSM...\n")

parameters_to_check = [
    "/app/lakehouse-agent/mcp-server-runtime-arn",
    "/app/lakehouse-agent/mcp-server-runtime-id",
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response["Parameter"]["Value"]
        print(f"✅ {param_name}")
        print(f"   Value: {value}")
    except ssm_client.exceptions.ParameterNotFound:
        print(f"❌ {param_name} - NOT FOUND")
        all_found = False
    except Exception as e:
        print(f"⚠️  {param_name} - ERROR: {e}")
        all_found = False

if all_found:
    print("\n✅ MCP Server Runtime configuration verified in SSM!")
else:
    print("\n⚠️  MCP Server Runtime parameters missing.")
    print("    agentcore_cli_deploy.py should have saved these automatically.")
    print("    Check the deployment output for errors.")

## Summary

✅ **MCP Server Deployment Complete!**

The MCP Server Runtime has been deployed and configuration saved to SSM Parameter Store.

**Next Steps:**
Run **05a-deploy-claims-gateway.ipynb** to deploy the AgentCore Claims Gateway (GW1)